In [23]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# -----------------------------
# 1. Small custom dataset
# -----------------------------
english_sentences = [
    "hello",
    "how are you",
    "i am fine",
    "what is your name",
    "my name is shyam",
    "thank you",
    "good morning",
    "good night",
    "i love programming",
    "see you later"
]

french_sentences = [
    "bonjour",
    "comment ca va",
    "je vais bien",
    "quel est ton nom",
    "mon nom est shyam",
    "merci",
    "bonjour",
    "bonne nuit",
    "j aime programmer",
    "a plus tard"
]

# Add start and end tokens to target sentences
french_sentences = ["start " + sent + " end" for sent in french_sentences]

# -----------------------------
# 2. Tokenization
# -----------------------------
eng_tokenizer = Tokenizer()
eng_tokenizer.fit_on_texts(english_sentences)
encoder_sequences = eng_tokenizer.texts_to_sequences(english_sentences)

fr_tokenizer = Tokenizer()
fr_tokenizer.fit_on_texts(french_sentences)
decoder_sequences = fr_tokenizer.texts_to_sequences(french_sentences)

eng_vocab_size = len(eng_tokenizer.word_index) + 1
fr_vocab_size = len(fr_tokenizer.word_index) + 1

max_encoder_len = max(len(seq) for seq in encoder_sequences)
max_decoder_len = max(len(seq) for seq in decoder_sequences)

encoder_input_data = pad_sequences(
    encoder_sequences,
    maxlen=max_encoder_len,
    padding="post"
)

decoder_input_data = []
decoder_target_data = []

for seq in decoder_sequences:
    decoder_input_data.append(seq[:-1])   # without end
    decoder_target_data.append(seq[1:])   # without start

decoder_input_data = pad_sequences(
    decoder_input_data,
    maxlen=max_decoder_len - 1,
    padding="post"
)

decoder_target_data = pad_sequences(
    decoder_target_data,
    maxlen=max_decoder_len - 1,
    padding="post"
)

# One-hot encode target output
decoder_target_data = tf.keras.utils.to_categorical(
    decoder_target_data,
    num_classes=fr_vocab_size
)

# -----------------------------
# 3. Encoder
# -----------------------------
latent_dim = 256

encoder_inputs = Input(shape=(max_encoder_len,))
encoder_embedding = tf.keras.layers.Embedding(
    eng_vocab_size,
    latent_dim,
    mask_zero=True
)(encoder_inputs)

encoder_lstm = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)

encoder_states = [state_h, state_c]

# -----------------------------
# 4. Decoder
# -----------------------------
decoder_inputs = Input(shape=(max_decoder_len - 1,))
decoder_embedding_layer = tf.keras.layers.Embedding(
    fr_vocab_size,
    latent_dim,
    mask_zero=True
)

decoder_embedding = decoder_embedding_layer(decoder_inputs)

decoder_lstm = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)

decoder_dense = Dense(fr_vocab_size, activation="softmax")
decoder_outputs = decoder_dense(decoder_outputs)

# -----------------------------
# 5. Full training model
# -----------------------------
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# -----------------------------
# 6. Train model
# -----------------------------
model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=2,
    epochs=500,
    verbose=1
)

Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_7       │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_8       │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 4, 256)    │      5,632 │ input_layer_7[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_3         │ (None, 4)         │          0 │ input_layer_7[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, 5, 256)    │      6,400 │ input_layer_8[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ [(None, 256),     │    525,312 │ embedding_2[0][0… │
│                     │ (None, 256),      │            │ not_equal_3[0][0] │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ [(None, 5, 256),  │    525,312 │ embedding_3[0][0… │
│                     │ (None, 256),      │            │ lstm_2[0][1],     │
│                     │ (None, 256)]      │            │ lstm_2[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 5, 25)     │      6,425 │ lstm_3[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,069,081 (4.08 MB)

 Trainable params: 1,069,081 (4.08 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.2286 - loss: 3.2090  
Epoch 2/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.2857 - loss: 3.1109
Epoch 3/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.2857 - loss: 2.9214
Epoch 4/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.2857 - loss: 2.6303
Epoch 5/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.2857 - loss: 2.4838
Epoch 6/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.2857 - loss: 2.4268
Epoch 7/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.2857 - loss: 2.2839
Epoch 8/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.2857 - loss: 2.1949
Epoch 9/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.2857 - loss: 2.0468
Epoch 10/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.3714 - loss: 1.8902
Epoch 11/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.3714 - loss: 1.6827
Epoch 12/500
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5143 - 

In [24]:
# -----------------------------
# 7. Inference models
# -----------------------------

# Encoder inference model
encoder_model = Model(encoder_inputs, encoder_states)

# Decoder inference model
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_single_input = Input(shape=(1,))

decoder_single_embedding = decoder_embedding_layer(decoder_single_input)

decoder_outputs, state_h, state_c = decoder_lstm(
    decoder_single_embedding,
    initial_state=decoder_states_inputs
)

decoder_states = [state_h, state_c]
decoder_outputs = decoder_dense(decoder_outputs)

decoder_model = Model(
    [decoder_single_input] + decoder_states_inputs,
    [decoder_outputs] + decoder_states
)

reverse_fr_word_index = {
    index: word for word, index in fr_tokenizer.word_index.items()
}

def translate_sentence(input_sentence):
    input_seq = eng_tokenizer.texts_to_sequences([input_sentence])
    input_seq = pad_sequences(input_seq, maxlen=max_encoder_len, padding="post")

    states_value = encoder_model.predict(input_seq, verbose=0)

    start_token = fr_tokenizer.word_index["start"]
    end_token = fr_tokenizer.word_index["end"]

    target_seq = np.array([[start_token]])

    translated_sentence = []

    for _ in range(max_decoder_len):
        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value,
            verbose=0
        )

        sampled_token_index = np.argmax(output_tokens[0, -1, :])

        if sampled_token_index == end_token:
            break

        sampled_word = reverse_fr_word_index.get(sampled_token_index, "")

        if sampled_word not in ["start", "end", ""]:
            translated_sentence.append(sampled_word)

        target_seq = np.array([[sampled_token_index]])
        states_value = [h, c]

    return " ".join(translated_sentence)

In [25]:
print(translate_sentence("hello"))
print(translate_sentence("how are you"))
print(translate_sentence("i am fine"))
print(translate_sentence("good night"))
print(translate_sentence("see you later"))

bonjour
comment ca va
je vais bien
bonne nuit
a plus tard
